# WhaleWisdom 2.0
A clean, auditable investment-research pipeline.

**Flow:** Ideas → Validation → Health Gates → Quality/Value/Momentum/Whale Scores → Risk Penalties → Final Rank → Portfolio


In [ ]:
# 1. Configuration
from dataclasses import dataclass
from typing import Dict, List
import numpy as np
import pandas as pd

@dataclass(frozen=True)
class Config:
    min_market_cap: float = 50_000_000
    min_piotroski: int = 5
    max_position: float = 0.10
    max_sector: float = 0.25
    min_portfolio_score: float = 75.0
    portfolio_value: float = 100_000

CFG = Config()
SCORE_WEIGHTS = {'quality':0.35,'valuation':0.25,'whale':0.20,'momentum':0.15,'health':0.05}


In [ ]:
# 2. Imports / data provider
# Install once in Colab if needed: !pip -q install yfinance
import yfinance as yf
from datetime import datetime

RUN_DATE = pd.Timestamp.utcnow().date()
print('Run date:', RUN_DATE)


In [ ]:
# 3. Idea sources — preserve provenance
IDEA_SOURCES: Dict[str, List[str]] = {
    'whale_manual': ['MSFT','AAPL','AMZN','GOOGL','NVDA','JPM','META','BRK-B','LLY','AVGO'],
    'alpha_manual': ['PYPL','ALL','QTWO','CLS','CCL','POWL','SFM','APP','UBER','CRM'],
}

def normalize_ticker(t):
    return str(t).upper().strip().replace('.', '-')

rows=[]
for source, tickers in IDEA_SOURCES.items():
    for t in tickers:
        rows.append((normalize_ticker(t), source))
ideas = pd.DataFrame(rows, columns=['ticker','source'])
provenance = ideas.groupby('ticker')['source'].agg(lambda x: sorted(set(x))).reset_index()
universe = provenance['ticker'].tolist()
display(provenance)


In [ ]:
# 4. Collect and validate market/fundamental data
def safe_get(d, key, default=np.nan):
    try:
        v = d.get(key, default)
        return default if v is None else v
    except Exception:
        return default

def collect_snapshot(ticker):
    s = yf.Ticker(ticker)
    info = s.info or {}
    hist = s.history(period='1y', auto_adjust=True)
    if hist.empty or 'Close' not in hist:
        return {'ticker':ticker,'data_ok':False,'data_issue':'missing price history'}
    close = hist['Close'].dropna()
    ret = close.pct_change().dropna()
    ma50 = close.rolling(50).mean().iloc[-1] if len(close)>=50 else np.nan
    ma200 = close.rolling(200).mean().iloc[-1] if len(close)>=200 else np.nan
    return {
        'ticker':ticker,'data_ok':True,'data_issue':'',
        'price':float(close.iloc[-1]),
        'market_cap':safe_get(info,'marketCap'),
        'sector':safe_get(info,'sector','Unknown'),
        'roe':safe_get(info,'returnOnEquity'),
        'profit_margin':safe_get(info,'profitMargins'),
        'revenue_growth':safe_get(info,'revenueGrowth'),
        'earnings_growth':safe_get(info,'earningsGrowth'),
        'debt_to_equity':safe_get(info,'debtToEquity'),
        'free_cashflow':safe_get(info,'freeCashflow'),
        'operating_cashflow':safe_get(info,'operatingCashflow'),
        'forward_pe':safe_get(info,'forwardPE'),
        'peg':safe_get(info,'pegRatio'),
        'beta':safe_get(info,'beta'),
        'mom_12m':float(close.iloc[-1]/close.iloc[0]-1) if len(close)>1 else np.nan,
        'volatility':float(ret.std()*np.sqrt(252)) if len(ret)>10 else np.nan,
        'above_50dma': bool(close.iloc[-1] > ma50) if pd.notna(ma50) else np.nan,
        'above_200dma': bool(close.iloc[-1] > ma200) if pd.notna(ma200) else np.nan,
        'golden_trend': bool(ma50 > ma200) if pd.notna(ma50) and pd.notna(ma200) else np.nan,
    }

snapshots = pd.DataFrame([collect_snapshot(t) for t in universe]).merge(provenance,on='ticker',how='left')
display(snapshots.head())


In [ ]:
# 5. Health gates — missing data is visible, never silently replaced with zero
def health_gate(r):
    reasons=[]
    if not r.get('data_ok',False): reasons.append(r.get('data_issue','data error'))
    if pd.notna(r.get('market_cap')) and r['market_cap'] < CFG.min_market_cap: reasons.append('market cap below floor')
    if pd.notna(r.get('operating_cashflow')) and r['operating_cashflow'] <= 0: reasons.append('non-positive operating cash flow')
    if pd.notna(r.get('free_cashflow')) and r['free_cashflow'] <= 0: reasons.append('non-positive free cash flow')
    return len(reasons)==0, '; '.join(reasons)

g = snapshots.apply(health_gate,axis=1,result_type='expand')
snapshots[['health_pass','health_reasons']] = g


In [ ]:
# 6. Transparent percentile scoring
def pct_score(s, higher_is_better=True):
    x = pd.to_numeric(s, errors='coerce')
    score = x.rank(pct=True, method='average')*100
    return score if higher_is_better else 100-score

df = snapshots.copy()
quality_parts = pd.DataFrame({
    'roe':pct_score(df['roe']),
    'margin':pct_score(df['profit_margin']),
    'rev_growth':pct_score(df['revenue_growth']),
    'earn_growth':pct_score(df['earnings_growth']),
})
df['quality'] = quality_parts.mean(axis=1, skipna=True)

value_parts = pd.DataFrame({
    'forward_pe':pct_score(df['forward_pe'],False),
    'peg':pct_score(df['peg'],False),
})
df['valuation'] = value_parts.mean(axis=1, skipna=True)

momentum_parts = pd.DataFrame({
    'mom12':pct_score(df['mom_12m']),
    'above50':df['above_50dma'].map({True:100,False:0}),
    'above200':df['above_200dma'].map({True:100,False:0}),
    'trend':df['golden_trend'].map({True:100,False:0}),
})
df['momentum'] = momentum_parts.mean(axis=1, skipna=True)

# Temporary whale proxy: source membership only. Replace in Phase 2 with quarter-aware 13F conviction.
df['whale'] = df['source'].apply(lambda xs: 70.0 if 'whale_manual' in xs else 30.0)
df['health'] = df['health_pass'].map({True:100.0,False:0.0})


In [ ]:
# 7. Explicit risk penalties and final conviction
def risk_penalty(r):
    p=0.0
    if pd.notna(r.get('beta')) and r['beta']>1.6: p += 8
    if pd.notna(r.get('volatility')) and r['volatility']>0.50: p += 8
    if pd.notna(r.get('forward_pe')) and r['forward_pe']>50: p += 6
    if not r.get('health_pass',False): p += 25
    return p

df['risk_penalty'] = df.apply(risk_penalty,axis=1)
df['base_score'] = sum(df[k].fillna(50)*w for k,w in SCORE_WEIGHTS.items())
df['final_score'] = (df['base_score'] - df['risk_penalty']).clip(0,100)
df['decision'] = pd.cut(df['final_score'],[-1,50,65,75,85,101],labels=['Reject','Research','Watch','Buy','Strong Buy'],right=False)
ranked = df.sort_values('final_score',ascending=False).reset_index(drop=True)
ranked.index += 1
cols=['ticker','source','sector','quality','valuation','whale','momentum','health','risk_penalty','final_score','decision','health_reasons']
display(ranked[cols].round(2))


In [ ]:
# 8. Deterministic portfolio construction before any optimizer
eligible = ranked[(ranked['final_score']>=CFG.min_portfolio_score) & (ranked['health_pass'])].copy()
if eligible.empty:
    portfolio = pd.DataFrame(columns=['ticker','sector','final_score','weight','dollars'])
else:
    raw = eligible['final_score'] / eligible['final_score'].sum()
    eligible['weight'] = raw.clip(upper=CFG.max_position)
    # Renormalize only within current candidates; sector-cap enforcement is the next refinement.
    if eligible['weight'].sum() > 0:
        eligible['weight'] = eligible['weight']/eligible['weight'].sum()
    eligible['dollars'] = eligible['weight']*CFG.portfolio_value
    portfolio = eligible[['ticker','sector','final_score','weight','dollars']]
display(portfolio.round({'final_score':2,'weight':4,'dollars':2}))


In [ ]:
# 9. Export an auditable run
ranked.reset_index(names='rank').to_csv('whalewisdom_ranked.csv',index=False)
portfolio.to_csv('whalewisdom_portfolio.csv',index=False)
print('Wrote whalewisdom_ranked.csv and whalewisdom_portfolio.csv')

# Phase 2 TODO: replace whale_manual proxy with live, timestamped 13F/Whale ingestion.
# Phase 3 TODO: point-in-time backtest against SPY before tuning score weights.


## Phase 2 — SEC 13F Whale Conviction

This section uses the SEC's flattened quarterly Form 13F data sets. CUSIP is the canonical security key. Ticker mapping is explicit enrichment rather than fuzzy matching.

**Current complete comparison:** 2026-06-30 vs 2026-03-31.


In [ ]:
# 10. Load SEC 13F whale engine
import os
from whale_13f import (
    SEC_13F_DATASETS, DEFAULT_MANAGERS,
    load_quarter, build_whale_scores, attach_ticker_mapping
)

# SEC expects an identifying User-Agent. Set this in Colab Secrets/environment.
# Example format only — do not hard-code credentials in the notebook.
SEC_USER_AGENT = os.environ.get('SEC_USER_AGENT')
if not SEC_USER_AGENT:
    print('Set SEC_USER_AGENT before running the download cells.')

display(DEFAULT_MANAGERS)


In [ ]:
# 11. Download the two complete SEC quarterly data sets and calculate whale scores
# These ZIPs are large (~95 MB each), so run this cell only when refreshing 13F data.
current_13f = load_quarter(SEC_13F_DATASETS['2026Q2'], user_agent=SEC_USER_AGENT)
previous_13f = load_quarter(SEC_13F_DATASETS['2026Q1'], user_agent=SEC_USER_AGENT)

whale_scores_cusip, whale_changes = build_whale_scores(
    current_13f,
    previous_13f,
    managers=DEFAULT_MANAGERS,
    current_period='2026-06-30',
    previous_period='2026-03-31',
)

display(whale_scores_cusip.head(30))


In [ ]:
# 12. Explicit CUSIP -> ticker enrichment
# Create/maintain this mapping separately. Required columns: CUSIP,ticker
MAPPING_FILE = 'cusip_ticker_mapping.csv'

if os.path.exists(MAPPING_FILE):
    ticker_map = pd.read_csv(MAPPING_FILE, dtype=str)
    whale_scores = attach_ticker_mapping(whale_scores_cusip, ticker_map)
    display(whale_scores.head(30))
else:
    whale_scores = whale_scores_cusip.copy()
    print(f'{MAPPING_FILE} not found. Whale scores remain keyed by CUSIP.')


In [ ]:
# 13. Replace the temporary manual whale proxy when mapped SEC scores are available
if 'ticker' in whale_scores.columns:
    sec_whale = whale_scores[['ticker','whale_score','whale_score_status']].dropna(subset=['ticker']).copy()
    sec_whale['ticker'] = sec_whale['ticker'].astype(str).str.upper().str.replace('.', '-', regex=False)
    df = df.drop(columns=['whale'], errors='ignore').merge(sec_whale, on='ticker', how='left')
    # Neutral score for names with no evidence from the selected manager set.
    df['whale'] = df['whale_score'].fillna(50.0)
    print('SEC 13F Whale score is now feeding the final conviction model.')
else:
    print('Ticker enrichment is required before SEC Whale scores can join the stock model.')
